In [ ]:
# Check if using Google Colab
try:
    from google.colab import drive

    USE_COLAB = True

    print("Running in Google Colab environment.")
except ImportError:
    USE_COLAB = False
    print("Running in local environment.")

In [ ]:
## Imports

import numpy as np
import os
import pandas as pd

import matplotlib.pyplot as plt

import datetime as dt

import warnings

warnings.filterwarnings(
    "ignore",
    message=r".*datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
    module=r"jupyter_client\.session",
)

warnings.filterwarnings(
    "ignore", category=DeprecationWarning, message=".*multi-threaded, use of fork().*"
)

from dotenv import load_dotenv

try: 
    import mne
    from mne_bids import BIDSPath

    from fooof import FOOOFGroup
    from fooof.sim.gen import gen_aperiodic

except ImportError as err:
    if USE_COLAB:
        # mute deprecation warnings from jupyter_client.session about datetime.datetime.utcnow() being deprecated
        
        
        print("Libraries not found. Installing libraries...")
        !pip install mne mne-bids pyvista pyvistaqt python-picard mne-icalabel pyprep fooof h5io --quiet


        import mne
        from mne_bids import BIDSPath
        from fooof import FOOOFGroup
        from fooof.sim.gen import gen_aperiodic


        print("Libraries loaded on colab")

    elif not USE_COLAB:
        print(f"Libraries not found on local machine: {err}")

if not USE_COLAB:
    mne.viz.set_3d_backend("pyvistaqt")
    %matplotlib qt
elif USE_COLAB:
    mne.viz.set_3d_backend("notebook")
    %matplotlib inline

In [ ]:
if USE_COLAB:
    # Access Google Drive files

    from google.colab import drive
    import os

    # get data
    GOOGLE_ROOT = "/content/drive/MyDrive/research_data/mind-wandering/Brandmeyer/"
    drive.mount("/content/drive")
    os.chdir(GOOGLE_ROOT)

    print(f"Current Directory: {os.getcwd()}")

In [ ]:
## Settings and Info

# Paths
load_dotenv()  # Load environment variables from .env file
ROOT = os.getenv("ROOT")
if USE_COLAB:
    ROOT = GOOGLE_ROOT

RAW_DATA = ROOT
DERIVATIVES = f"{ROOT}/derivatives"
DATA_FOLDER = f"{DERIVATIVES}/noica"
PREPROC_DATA = f"{DATA_FOLDER}/preprocessed"
EPOCHED_DATA = f"{DATA_FOLDER}/epoched"

# Analysis paths
ANALYSIS_PATH = f"{DATA_FOLDER}/analysis"  # Base path for all analyses
# Intermediary analysis file paths
PSD_PATH = f"{ANALYSIS_PATH}/PSD"
ZSCORE_PATH = f"{ANALYSIS_PATH}/Zscore"
FOOOF_PATH = f"{ANALYSIS_PATH}/FOOOF"

# Feature paths
FEATURES_PATH = f"{DATA_FOLDER}/features"

os.makedirs(ANALYSIS_PATH, exist_ok=True)  # ensure the directory exists
os.makedirs(PSD_PATH, exist_ok=True)  # ensure the directory exists
os.makedirs(ZSCORE_PATH, exist_ok=True)  # ensure the directory exists
os.makedirs(FOOOF_PATH, exist_ok=True)  # ensure the directory exists
os.makedirs(FEATURES_PATH, exist_ok=True)  # ensure the directory exists

N_JOBS = 1  # Num of cores to use for parallel processing (adjust as needed)

# Frequency bands
BANDS = {
    "delta": [2, 4],
    "theta": [4, 8],
    "alpha": [8, 12],
    "beta": [12, 30],
    "gamma": [30, 50],
}

SUBJECTS = [f"{i:03d}" for i in range(1, 25)]  # '001' through '024'
SESSIONS = ["01", "02", "03"]

In [ ]:
def get_band_avg(freqs, flattened_psd, band_range):
    # Find indices for the frequency band
    idx = np.logical_and(freqs >= band_range[0], freqs <= band_range[1])
    # Average the power (still in log scale, or convert back to linear)
    avg_log_amp = np.mean(flattened_psd[idx])
    return avg_log_amp

In [ ]:
### Run FOOOF and save the results for each subject and session

l_freq = 2
h_freq = 40
peak_width_limits = [1.0, 8.0]
max_n_peaks = 12
min_peak_height = 0.05
peak_threshold = 2
aperiodic_mode = "fixed"  # 'fixed' or 'knee'


def save_features(session, subject, output_folder):
    try:
        os.makedirs(output_folder, exist_ok=True)  # ensure the directory exists

        ## Load the PSD data
        bp = BIDSPath(
            subject=subject,
            session=session,
            task="meditation",
            description="epo",  # Identifies these as epochs
            suffix="psds",  # Official BIDS suffix
            datatype="eeg",
            root=PSD_PATH,  # Point to your derivatives folder
            extension=".h5",
            check=False,  # Allows us to define custom 'desc' tags
        )
        # Load psd data
        psds = mne.time_frequency.read_spectrum(bp.fpath)
        psds_data = psds.get_data()  # Shape: (n_epochs, n_channels, n_freqs)
        # Load metadata for the epochs (probes for meditation depth, mind wandering depth, fatigue)
        probe_cols = ["meditation_depth", "mw_depth", "fatigue"]
        probe_df = (
            psds.metadata.loc[:, probe_cols]
            .copy()
            .astype(int)
            .reset_index()  # Resetting index means epochs wont be aligned with the original epoch indices (eg. it would sometimes skips from 1 to 4)
        )

        fg = FOOOFGroup(
            peak_width_limits=peak_width_limits,
            max_n_peaks=max_n_peaks,
            min_peak_height=min_peak_height,
            peak_threshold=peak_threshold,
            aperiodic_mode=aperiodic_mode,
            verbose=False,  # Suppress console output
        )

        features = []  # List to hold results for all channels and epochs
        for epoch_idx, epoch_psd in enumerate(psds_data):
            fg.fit(
                freqs=psds.freqs,
                power_spectra=epoch_psd,  # Shape: (n_channels, n_freqs)
                freq_range=(l_freq, h_freq),
                n_jobs=N_JOBS,
            )

            aps = fg.get_params("aperiodic_params")  # Array of shape (n_channels, 2)
            rsquared = fg.get_params("r_squared")  # Array of shape (n_channels,)
            errors = fg.get_params("error")  # Array of shape (n_channels,)

            all_powers = fg.power_spectra
            freqs = fg.freqs

            for ch_idx, ch_name in enumerate(psds.ch_names):
                # fm = fg.get_fooof(ind=ch_idx)  # single FOOOF model for this channel
                # # flattened spectrum in log10 space
                # # (avoid relying on group internals; use model internals per fitted model)
                # flattened_psd = fm.power_spectrum - fm._ap_fit

                ap_params = aps[ch_idx]  # Get the params for this channel
                ap_fit = gen_aperiodic(freqs, ap_params)

                flattened_psd = all_powers[ch_idx] - ap_fit

                row = {
                    "subject": subject,
                    "session": session,
                    "channel": ch_name,
                    "epoch_idx": epoch_idx,
                    probe_cols[0]: int(probe_df.loc[epoch_idx, probe_cols[0]]),
                    probe_cols[1]: int(probe_df.loc[epoch_idx, probe_cols[1]]),
                    probe_cols[2]: int(probe_df.loc[epoch_idx, probe_cols[2]]),
                    "aperiodic_offset": float(aps[ch_idx][0]),
                    "aperiodic_exponent": float(aps[ch_idx][1]),
                    "r_squared": float(rsquared[ch_idx]),
                    "error": float(errors[ch_idx]),
                }

                # Calculate mean power for each band using the original spectrum
                for band_name, band_limits in BANDS.items():
                    row[f"{band_name}_mean_power"] = float(
                        get_band_avg(freqs, epoch_psd[ch_idx], band_limits)
                    )

                features.append(row)  # Add the row to the features lis

                # Calculate mean power for each band using the FOOOF-flattened spectrum
                for band_name, band_limits in BANDS.items():
                    row[f"{band_name}_flat_mean_power"] = float(
                        get_band_avg(freqs, flattened_psd, band_limits)
                    )

                features.append(row)  # Add the row to the features list

        # Save features to TSV file
        df_features = pd.DataFrame(features)
        file_name = f"sub-{subject}_ses-{session}_features.tsv"
        out_path = os.path.join(output_folder, file_name)
        df_features.to_csv(out_path, sep="\t", index=False)  # Save file

        kb = os.path.getsize(out_path) // (1000)
        ts = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        message = f"{ts} ✅ Saved features for Sub-{subject} Ses-{session} ({kb} KB)"
        print(message)
        return message

    except Exception as e:
        ts = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        message = f"{ts} ❌ Failed on Sub-{subject} Ses-{session}: {e}"
        print(message)
        return message

    finally:
        # Make derivative dataset desc
        desc = {
            "Name": "EEG Features File for Mind Wandering Detection Thesis",
            "BIDSVersion": "1.8.0",
            "DatasetType": "derivative",
        }
        with open(f"{output_folder}/dataset_description.json", "w") as f:
            import json

            json.dump(desc, f, indent=4)


# Run FOOOF for each subject and session
for subject in SUBJECTS:
    for session in SESSIONS:
        save_features(session, subject, f"{FEATURES_PATH}")